# ML Coding Interviews: Overview

ML coding rounds are distinct from LeetCode rounds: interviewers care about vectorized math, numerical stability, and clean ML abstractions — not graph algorithms or dynamic programming. This note covers interview formats, grading rubrics, how to allocate your 35 minutes, and a NumPy vectorization refresher that everything downstream depends on.

## What Interviewers Test
- Can you implement ML primitives (losses, layers, optimizers) without library crutches?
- Do you vectorize correctly, or do you write Python loops?
- Do you know when your code would break numerically (overflow, underflow, divide-by-zero)?
- Can you reason about time/space complexity of ML operations?
- Do you communicate intent before coding (explain your plan, handle edge cases aloud)?
- Do you verify your implementation (print shapes, compare to sklearn, spot-check gradients)?

## Interview Formats

| Format | Description | Example Task |
|---|---|---|
| **Implement from scratch** | Build a well-known primitive in NumPy/PyTorch | "Implement batch normalization" |
| **Debug broken code** | Find and fix bugs in a ~50-line ML snippet | Wrong gradient sign, missing normalization |
| **Extend a baseline** | Given a working model, add a feature | Add L2 reg to an existing linear model |

> 💡 **Interview Tip:** Before writing a single line, say: *"I'll start with the math, then code the forward pass, then test it."* Interviewers grade communication as heavily as correctness.

## Grading Rubric (FAANG-style)

| Criterion | What they look for |
|---|---|
| **Correctness** | Produces right outputs on the happy path + edge cases |
| **Vectorization** | No Python `for` loops over samples/features; uses NumPy broadcasting |
| **Numerical stability** | Aware of log(0), exp overflow, division by tiny numbers |
| **Complexity** | Can state time/space complexity of the key operations |
| **Communication** | Explains approach, narrates tradeoffs, asks clarifying questions |
| **Verification** | Checks output shapes, compares to reference, adds an assertion |

> 💡 **Interview Tip:** If stuck, say the naive O(n²) loopy version first, then optimize. Showing you can recognize and fix the bottleneck is often more impressive than jumping to the optimal solution silently.

## How to Allocate 35 Minutes

```
 0–3 min  Clarify: input types, shapes, edge cases ("Is batch size always fixed?")
 3–8 min  Outline: write pseudocode or the math in comments before coding
 8–25 min Implement: vectorized, clean, commented
25–30 min Verify: print shapes, small numeric example, compare to reference
30–35 min Discuss: complexity, extensions, failure modes
```

Skipping the outline phase is the #1 mistake — interviewers hate watching someone type and delete.

## NumPy Vectorization Crash Refresher

### Broadcasting Rules
NumPy aligns shapes from the **right**. Dimensions of size 1 are stretched to match.

```
(3, 1) + (1, 4) → (3, 4)   ✓ valid
(3, 2) + (4,)   → ERROR    ✗ trailing dim 2 ≠ 4
```

### Axis Semantics (the thing people always get wrong)
- `axis=0`: reduce **across rows** (collapse rows, keep columns) — e.g., mean per feature
- `axis=1`: reduce **across columns** (collapse columns, keep rows) — e.g., mean per sample



In [ ]:
import numpy as np
np.random.seed(42)

# --- Broadcasting ---
X = np.random.randn(100, 10)  # 100 samples, 10 features
mu = X.mean(axis=0)           # (10,) — mean per feature
X_centered = X - mu           # (100, 10) - (10,) → broadcast OK

# --- Axis semantics ---
print("mean per feature (axis=0):", X.mean(axis=0).shape)   # (10,)
print("mean per sample  (axis=1):", X.mean(axis=1).shape)   # (100,)

# --- keepdims for safe broadcasting in subsequent ops ---
norms = np.linalg.norm(X, axis=1, keepdims=True)  # (100, 1)
X_normalized = X / norms                           # (100, 10) — safe

# --- einsum basics ---
A = np.random.randn(4, 5)
B = np.random.randn(5, 3)
C_matmul  = np.einsum('ij,jk->ik', A, B)          # standard matmul (4,3)
C_batched = np.einsum('bij,bjk->bik',
    np.random.randn(8,4,5),
    np.random.randn(8,5,3))                        # batched matmul (8,4,3)
trace     = np.einsum('ii->', np.random.randn(5,5))  # trace

print("matmul shape:", C_matmul.shape)
print("batched matmul shape:", C_batched.shape)
print("trace:", trace)


## Warm-Up: Numerically Stable Softmax

This is the single most common numerical-stability question. The naive version overflows with large logits; the stable version subtracts the max first.

**Math:** $\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$

**Stable trick:** $\frac{e^{z_i - c}}{\sum_j e^{z_j - c}}$ for any constant $c$ — the constant cancels. Choose $c = \max(z)$ to prevent overflow.



In [ ]:
import numpy as np

# ---------- Naive softmax ----------
def softmax_naive(z):
    """Overflows for large logits."""
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=-1, keepdims=True)

# ---------- Stable softmax ----------
def softmax_stable(z):
    """Subtract max before exp. Mathematically identical, numerically safe."""
    z = z - z.max(axis=-1, keepdims=True)   # shift so max is 0
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=-1, keepdims=True)

# ---------- Demonstrate the overflow ----------
large_logits = np.array([1000.0, 1001.0, 1002.0])

print("=== Naive softmax on large logits ===")
result_naive = softmax_naive(large_logits)
print(f"  exp values: {np.exp(large_logits)}")  # inf inf inf
print(f"  softmax:    {result_naive}")           # nan nan nan

print()
print("=== Stable softmax on large logits ===")
result_stable = softmax_stable(large_logits)
print(f"  shifted logits: {large_logits - large_logits.max()}")
print(f"  softmax:        {result_stable}")      # correct probabilities

# ---------- Verify they match on normal inputs ----------
normal = np.random.randn(4, 5)
assert np.allclose(softmax_naive(normal), softmax_stable(normal)), "Should match!"
print()
print("On normal inputs both versions match ✓")

# ---------- Batched version ----------
batch = np.random.randn(32, 10)  # 32 samples, 10 classes
probs = softmax_stable(batch)
print(f"\nBatch softmax shape: {probs.shape}")
print(f"Each row sums to 1: {np.allclose(probs.sum(axis=1), 1.0)}")


## Common Interview Questions

**Q: Why does softmax overflow and how do you fix it?**
`exp(z)` grows without bound, so large logits like 1000 produce `inf`. Subtracting `max(z)` before `exp` shifts values to ≤ 0, bounding the result between 0 and 1. The constant cancels in numerator and denominator, so the output is mathematically unchanged.

**Q: What's the time complexity of a matrix multiply of (n, d) × (d, m)?**
O(n·d·m) — each of the n·m output elements requires a d-element dot product. In practice, BLAS routines achieve close to peak FLOP/s by exploiting cache locality and SIMD.

**Q: Why avoid Python loops over samples in ML code?**
NumPy operations run in compiled C with SIMD vectorization; a Python loop over 10,000 samples adds ~10,000× interpreter overhead. Even worse, it prevents broadcasting and often signals a misunderstanding of the underlying math.

**Q: What is `keepdims=True` for?**
Without it, reducing a (100, 10) array along axis=0 gives shape (10,), which can't broadcast back against (100, 10). With `keepdims=True` you get (1, 10), which broadcasts cleanly.

**Q: When would you use `einsum` vs `@` or `np.matmul`?**
`einsum` is clearest for non-standard contractions: batch outer products, trace, multi-index contractions in attention. For vanilla matmul, `@` is more readable. `einsum` also makes shape intent explicit, which helps catch bugs in complex tensor operations.

**Q: What's the difference between `np.dot` and `np.matmul`?**
For 2D arrays they're equivalent. `np.dot` with 1D inputs does an inner product and returns a scalar; `np.matmul` treats 1D as a vector and returns a vector. For >2D, `np.matmul` broadcasts over leading batch dimensions while `np.dot` sums over the last axis of the first arg and second-to-last of the second — confusing and bug-prone. Prefer `@` or `np.matmul` for clarity.

## Key Takeaways
- ML coding rounds grade vectorization, numerical stability, and communication — not algorithm tricks
- Always state your plan before coding; interviewers grade the process
- Axis=0 reduces across rows (gives per-column result); axis=1 reduces across columns (per-row)
- Broadcasting aligns shapes from the right; `keepdims=True` preserves rank for safe follow-on ops
- Stable softmax subtracts `max(z)` before `exp` — this cancels algebraically but prevents `inf`
- Verify implementations: check shapes, run a small numeric example, compare to sklearn
- `einsum` makes contraction intent explicit and catches shape bugs early